# Demand Forecasting

This notebook focuses on developing machine learning models for forecasting electricity demand in Germany. The engineered dataset created in the previous notebook serves as the foundation for model training, evaluation, and comparison.

## Loading the Engineered Dataset

The engineered dataset containing demand, weather, renewable energy, temporal, and lag-based features was loaded for forecasting analysis.

In [10]:
import pandas as pd

model_df = pd.read_csv(
    "../data/processed/master_features.csv"
)

print(model_df.shape)

model_df.head()

(50232, 29)


,utc_timestamp,DE_load_actual_entsoe_transparency,DE_solar_generation_actual,DE_wind_generation_actual,DE_solar_capacity,DE_wind_capacity,DE_LU_price_day_ahead,time,temperature_C,humidity_pct,...,week_of_year,season,renewable_generation,renewable_share_pct,load_lag_1,load_lag_24,load_lag_168,rolling_mean_24,hour_sin,hour_cos
0,2015-01-08 00:00:00+00:00,48041.0,0.0,15807.0,37258.0,27966.0,NaN,2015-01-08 00:00:00+00:00,1.56,86.8,...,2,Winter,15807.0,32.903145,50460.0,45125.0,41151.0,60234.500000,0.000000,1.000000
1,2015-01-08 01:00:00+00:00,47074.0,0.0,16449.0,37258.0,27966.0,NaN,2015-01-08 01:00:00+00:00,1.70,86.0,...,2,Winter,16449.0,34.942856,48041.0,44217.0,40135.0,60353.541667,0.258819,0.965926
2,2015-01-08 02:00:00+00:00,47228.0,0.0,16695.0,37258.0,27966.0,NaN,2015-01-08 02:00:00+00:00,1.90,84.0,...,2,Winter,16695.0,35.349792,47074.0,44368.0,39106.0,60472.708333,0.500000,0.866025
3,2015-01-08 03:00:00+00:00,48253.0,0.0,16787.0,37258.0,27966.0,NaN,2015-01-08 03:00:00+00:00,2.20,83.6,...,2,Winter,16787.0,34.789547,47228.0,45298.0,38765.0,60595.833333,0.707107,0.707107
4,2015-01-08 04:00:00+00:00,51155.0,0.0,16751.0,37258.0,27966.0,NaN,2015-01-08 04:00:00+00:00,2.42,84.2,...,2,Winter,16751.0,32.745577,48253.0,48598.0,38941.0,60702.375000,0.866025,0.500000


### Dataset Overview

The engineered forecasting dataset was successfully loaded for model development. The dataset contains 50,232 observations and 29 variables, including electricity demand, weather conditions, renewable energy metrics, temporal indicators, lagged demand features, rolling statistics, and cyclical time representations.

The dataset represents the final output of the feature engineering workflow and serves as the foundation for training and evaluating electricity demand forecasting models.

## Data Quality Verification

Before model development, the dataset was inspected to verify data types and identify any remaining missing values that could affect model training and evaluation.

In [11]:
model_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50232 entries, 0 to 50231
Data columns (total 29 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   utc_timestamp                       50232 non-null  str    
 1   DE_load_actual_entsoe_transparency  50232 non-null  float64
 2   DE_solar_generation_actual          50232 non-null  float64
 3   DE_wind_generation_actual           50232 non-null  float64
 4   DE_solar_capacity                   50232 non-null  float64
 5   DE_wind_capacity                    50232 non-null  float64
 6   DE_LU_price_day_ahead               17540 non-null  float64
 7   time                                50232 non-null  str    
 8   temperature_C                       50232 non-null  float64
 9   humidity_pct                        50232 non-null  float64
 10  cloud_cover_pct                     50232 non-null  float64
 11  wind_speed_ms                       50232 non-null  

### Data Quality Verification

The forecasting dataset was inspected to verify data completeness and variable types. All demand, weather, renewable energy, temporal, lagged, rolling, and cyclical features contained complete observations suitable for machine learning.

The electricity price variable contained substantial missing data, consistent with findings from the feature engineering stage. Consequently, this feature will be excluded from the initial forecasting models to preserve the full sample size. Timestamp variables were stored as strings following dataset export; however, their temporal information has already been captured through engineered calendar and cyclical features.

## Defining Target and Predictor Variables

The target variable was defined as electricity demand, while predictor variables consisted of weather, renewable energy, temporal, lagged, rolling, and cyclical features. Timestamp fields and electricity price data were excluded from the initial forecasting models.

In [12]:
target = 'DE_load_actual_entsoe_transparency'

features = [
    col for col in model_df.columns
    if col not in [
        'DE_load_actual_entsoe_transparency',
        'utc_timestamp',
        'time',
        'DE_LU_price_day_ahead'
    ]
]

X = model_df[features]
y = model_df[target]

print("Number of features:", len(features))
print("X shape:", X.shape)
print("y shape:", y.shape)

Number of features: 25
X shape: (50232, 25)
y shape: (50232,)


## Train-Test Split

Electricity demand forecasting is a time-series problem; therefore, the dataset was split chronologically rather than randomly. Earlier observations were used for training, while more recent observations were reserved for testing to simulate real-world forecasting conditions.

In [13]:
split_idx = int(len(model_df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 40185
Testing samples: 10047


### Time-Series Data Partitioning

The dataset was divided into training and testing subsets using an 80:20 chronological split. A total of 40,185 observations were used for model training, while 10,047 observations were reserved for out-of-sample evaluation.

By preserving temporal order, the split mimics real-world forecasting scenarios in which future electricity demand is predicted using only historical information.

## Baseline Forecast Model

A persistence forecasting approach was used as the baseline model. This method assumes that electricity demand at the current time step is equal to demand during the previous hour. Baseline performance provides a benchmark against which machine learning models can be evaluated.

In [14]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

baseline_pred = X_test['load_lag_1']

baseline_mae = mean_absolute_error(
    y_test,
    baseline_pred
)

baseline_rmse = root_mean_squared_error(
    y_test,
    baseline_pred
)

baseline_r2 = r2_score(
    y_test,
    baseline_pred
)

print("Baseline MAE :", round(baseline_mae, 2))
print("Baseline RMSE:", round(baseline_rmse, 2))
print("Baseline R²  :", round(baseline_r2, 4))

Baseline MAE : 1877.82
Baseline RMSE: 2461.99
Baseline R²  : 0.9374


### Baseline Forecast Performance

A persistence forecasting model was used as the baseline benchmark, where electricity demand at the current hour was assumed to be equal to demand during the previous hour.

The baseline model achieved an MAE of 1,877.82 MW, an RMSE of 2,461.99 MW, and an R² score of 0.9374. These results indicate strong temporal persistence in electricity demand, with more than 93% of demand variability explained by the previous hour's demand alone.

All subsequent machine learning models must outperform this baseline to demonstrate added predictive value.

## Identifying Non-Numeric Features

Machine learning algorithms such as linear regression require numerical input variables. The feature set was inspected to identify categorical variables that require encoding before model training.

In [15]:
X.dtypes[X.dtypes == 'object']

Series([], dtype: object)

### Feature Type Verification

Prior to model training, predictor variables were inspected to ensure compatibility with machine learning algorithms. Numerical models such as linear regression require all input features to be represented numerically. Therefore, any remaining non-numeric variables must be identified and appropriately encoded or removed before training.

In [16]:
X.select_dtypes(exclude=['number']).columns.tolist()

['season']

## Encoding Seasonal Information

The season variable is categorical and therefore cannot be directly processed by linear regression. One-hot encoding was applied to transform seasonal categories into numerical indicator variables while preserving seasonal information.

In [17]:
X = pd.get_dummies(
    X,
    columns=['season'],
    drop_first=True
)

print(X.shape)

(50232, 27)


### Seasonal Feature Encoding

The categorical season variable was transformed into numerical indicator variables using one-hot encoding. This process replaced a single non-numeric feature with multiple binary seasonal indicators, increasing the predictor set from 25 to 27 features.

The transformation preserves seasonal information while ensuring compatibility with machine learning algorithms such as linear regression. Because the predictor matrix changed after encoding, the train-test split was recreated to ensure that the updated feature set was consistently represented in both training and testing datasets.

## Updating the Modeling Dataset

Following seasonal feature encoding, the predictor matrix was updated and the chronological train-test split was recreated to ensure consistency during model training and evaluation.

In [19]:
split_idx = int(len(X) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print(X_train.shape)
print(X_test.shape)

(40185, 27)
(10047, 27)


### Updated Modeling Dataset

Following one-hot encoding of the seasonal feature, the predictor matrix increased from 25 to 27 variables. The train-test partition was subsequently recreated to ensure that the updated feature set was consistently represented in both training and testing subsets.

This step maintains the integrity of the forecasting workflow and ensures that all engineered features are available during model development and evaluation.

## Linear Regression Model

A multiple linear regression model was developed using the engineered feature set. The model serves as a simple yet interpretable machine learning baseline for evaluating the predictive value of the engineered features.

In [20]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()

lr_model.fit(
    X_train,
    y_train
)

lr_pred = lr_model.predict(X_test)

print("Linear Regression trained successfully.")

Linear Regression trained successfully.


## Evaluating Linear Regression Performance

The trained Linear Regression model was evaluated on the test dataset using Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and the coefficient of determination (R²). These metrics were compared directly against the persistence baseline to assess whether the engineered features provide additional predictive value.

In [21]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

lr_mae = mean_absolute_error(
    y_test,
    lr_pred
)

lr_rmse = root_mean_squared_error(
    y_test,
    lr_pred
)

lr_r2 = r2_score(
    y_test,
    lr_pred
)

print("Linear Regression MAE :", round(lr_mae, 2))
print("Linear Regression RMSE:", round(lr_rmse, 2))
print("Linear Regression R²  :", round(lr_r2, 4))

Linear Regression MAE : 1168.2
Linear Regression RMSE: 1490.44
Linear Regression R²  : 0.9771


### Linear Regression Performance

The Linear Regression model substantially outperformed the persistence baseline across all evaluation metrics. The model achieved an MAE of 1,168.20 MW, an RMSE of 1,490.44 MW, and an R² score of 0.9771.

Compared with the persistence forecast, the Linear Regression model reduced forecasting errors by approximately 38–40% while increasing explained variance from 93.74% to 97.71%. These results demonstrate that the engineered weather, renewable energy, temporal, lagged, rolling, and seasonal features provide substantial predictive value beyond simple demand persistence.

The strong performance also confirms the effectiveness of the feature engineering strategy developed in the previous notebook.

| Metric | Baseline | Linear Regression | Improvement |
| ------ | -------: | ----------------: | ----------: |
| MAE    |  1877.82 |       **1168.20** |  **-37.8%** |
| RMSE   |  2461.99 |       **1490.44** |  **-39.5%** |
| R²     |   0.9374 |        **0.9771** | **+0.0397** |


## Linear Regression Coefficient Analysis

Model coefficients were examined to identify the variables with the strongest positive and negative influence on electricity demand forecasts.

In [22]:
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_model.coef_
})

coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()

coef_df = coef_df.sort_values(
    'Abs_Coefficient',
    ascending=False
)

coef_df.head(15)

,Feature,Coefficient,Abs_Coefficient
22,hour_sin,1989.292717,1989.292717
23,hour_cos,-963.879840,963.879840
12,year,506.281383,506.281383
13,is_weekend,-274.447940,274.447940
26,season_Winter,-237.538493,237.538493
24,season_Spring,-171.509180,171.509180
17,renewable_share_pct,-143.079319,143.079319
10,day_of_week,-142.291776,142.291776
14,quarter,111.133881,111.133881
9,hour,-62.044195,62.044195


### Interpretation of Linear Regression Coefficients

The largest regression coefficients were associated with cyclical time variables, seasonal indicators, weekend effects, and renewable energy share. These results suggest that daily demand cycles and calendar-related patterns play an important role in explaining electricity demand variability.

However, coefficient magnitudes should be interpreted cautiously because predictor variables are measured on different scales. Variables such as lagged demand and renewable generation have much larger numerical ranges than cyclical features, making direct coefficient comparisons potentially misleading. A standardized coefficient analysis would provide a more reliable assessment of relative feature importance.

## Standardized Feature Importance

To enable meaningful comparison of feature effects, predictor variables were standardized prior to model fitting. Standardized coefficients were then examined to identify the variables with the strongest relative influence on electricity demand forecasts.

In [23]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_scaled = LinearRegression()

lr_scaled.fit(
    X_train_scaled,
    y_train
)

coef_scaled = pd.DataFrame({
    'Feature': X_train.columns,
    'Standardized_Coefficient': lr_scaled.coef_
})

coef_scaled['Abs_Coefficient'] = (
    coef_scaled['Standardized_Coefficient']
    .abs()
)

coef_scaled.sort_values(
    'Abs_Coefficient',
    ascending=False
).head(15)

,Feature,Standardized_Coefficient,Abs_Coefficient
18,load_lag_1,7833.460336,7833.460336
17,renewable_share_pct,-2432.850781,2432.850781
22,hour_sin,1406.666245,1406.666245
1,DE_wind_generation_actual,1380.012524,1380.012524
20,load_lag_168,1221.141651,1221.141651
16,renewable_generation,959.328000,959.328000
19,load_lag_24,882.846845,882.846845
23,hour_cos,-681.554381,681.554381
12,year,677.560693,677.560693
21,rolling_mean_24,-605.155745,605.155745


### Standardized Feature Importance

Standardized coefficients revealed that lagged demand variables are the most influential predictors of electricity demand. The previous hour's demand (**load_lag_1**) exhibited the strongest effect, confirming the high temporal persistence observed during exploratory analysis.

Other important predictors included renewable energy share, wind generation, weekly demand patterns (**load_lag_168**), and cyclical hourly features. The prominence of these variables indicates that electricity demand is driven by a combination of historical consumption patterns, renewable generation dynamics, and recurring temporal cycles.

The results validate the feature engineering strategy and confirm that the strongest forecasting signals originate from both demand history and renewable energy system behavior.